# PRISM Notebook 05: Alpha Backtest

Event-driven backtest with touch fills, Kalshi fees, block-bootstrap Sharpe CI.

In [ ]:
import pandas as pd
from src.data.database import PrismDatabase
from src.backtest.engine import PredictionMarketBacktester
from src.backtest.metrics import BacktestMetrics
from src.backtest.sizing import KellySizer
from src.models.inplay_xgb import XGBInPlayModel

db = PrismDatabase()
edges, prices, states = db.query_df('SELECT * FROM edge_signals'), db.query_df('SELECT * FROM market_prices'), db.query_df('SELECT * FROM game_states')
final = XGBInPlayModel.final_game_states(states)
outcomes = pd.DataFrame({'game_id': final.game_id, 'home_won': XGBInPlayModel.home_win_outcomes(final).astype(bool)})
meta = states[['game_id','game_date','sport','seconds_remaining']].drop_duplicates()
edges = edges.merge(meta, on=['game_id','seconds_remaining'], how='left')
results = PredictionMarketBacktester().run(edges, prices, outcomes, KellySizer())
metrics = BacktestMetrics().compute_all(results)
BacktestMetrics().save_plots(results)
metrics